# CAB V2 Compact-20 3-Model Provider Runbook

Project: Causal Agent Bench
Run name: Compact-20 provider pilot after gates
Prompt file: `05_COMPACT20_3MODEL_PREFLIGHT_STRICT.md`
Expected platform: Local approved provider machine
Expected accelerator: CPU/network
Internet: on only for approved provider API calls after gates
Required datasets: Human-reviewed Compact-20 locked slice and approved config
Required secrets: Provider credentials via environment variables only; never stored or printed
Expected disk usage: 5-20 GB
Expected RAM usage: 4-16 GB
Estimated runtime: see table below

This notebook is safe by default. Validation cells run locally in the notebook environment. Heavy execution cells contain the real commands but require `EXECUTE = True` after input validation passes on the intended platform.


## Estimated Runtime

        | Stage | Expected accelerator | Estimated runtime | Notes |
| ----- | -------------------- | ----------------: | ----- |
| Setup | CPU/network | 5-10 min | env and key presence booleans |
| Preflight | CPU | 10-30 min | config/cost/dry-run |
| Provider run | CPU/network | 30-90 min | only after approval |
| Audit/import | CPU | 10-30 min | evidence safety and claim ledger |

        Best case: 5-10 min.
        Expected runtime: 30-90 min.
        Worst case: 10-30 min.
        Runtime is mainly controlled by dataset size, model load time, accelerator availability, and shard retry count. Resume support is described in the project report and per-command comments.


In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ROOT = Path.cwd()
print('cwd=', ROOT)
print('python=', sys.version)
print('platform=', platform.platform())
print('disk_free_gb=', round(shutil.disk_usage(ROOT).free / 1e9, 2))
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('cuda_device_count=', torch.cuda.device_count())
    if torch.cuda.is_available():
        print('cuda_devices=', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as exc:
    print('torch_check_unavailable=', repr(exc))


In [ ]:
import random

try:
    import numpy as np
    np.random.seed(12345)
except Exception:
    pass
random.seed(12345)
os.environ.setdefault('PYTHONHASHSEED', '12345')
print('deterministic_seed=12345')


In [ ]:
required_paths = [
    "data/human_validation/compact20_real_review/HUMAN_TODO_EXACT_ROWS.csv",
    "docs/approvals/COMPACT20_3MODEL_LIVE_APPROVAL.md",
    "configs/compact20_3model_APPROVED.yaml",
    "scripts/check_evidence_safety.py",
]
missing = []
for raw in required_paths:
    p = ROOT / raw
    print(raw, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() and p.is_file() else '')
    if not p.exists():
        missing.append(raw)
if missing:
    print('MISSING_REQUIRED_PATHS=', missing)
else:
    print('input_validation=PASS')


In [ ]:
def run_cmd(cmd, timeout=None):
    print('\n$ ' + cmd)
    return subprocess.run(cmd, shell=True, check=True, timeout=timeout)


EXECUTE = False
LIVE_APPROVAL_VERIFIED = False
commands = [
    "python3 scripts/check_evidence_safety.py",
    "PYTHONPATH=src python3 -m causal_agent_bench validate-config --config configs/compact20_3model_APPROVED.yaml",
    "PYTHONPATH=src python3 -m causal_agent_bench plan-run --config configs/compact20_3model_APPROVED.yaml",
    "PYTHONPATH=src python3 -m causal_agent_bench estimate-run-cost --config configs/compact20_3model_APPROVED.yaml --output-dir /tmp/cab_compact20_3model_cost",
    "PYTHONPATH=src python3 -m causal_agent_bench dry-run --config configs/compact20_3model_APPROVED.yaml --output-dir /tmp/cab_compact20_3model_dryrun",
    "PYTHONPATH=src python3 -m causal_agent_bench run --config configs/compact20_3model_APPROVED.yaml",
]
print('Set EXECUTE=True only on the expected platform after validation passes.')
if EXECUTE and not LIVE_APPROVAL_VERIFIED:
    raise SystemExit('BLOCKED: live provider execution requires explicit approval')
if EXECUTE:
    for cmd in commands:
        run_cmd(cmd)
else:
    for cmd in commands:
        print('DRY_RUN_COMMAND:', cmd)


In [ ]:
output_candidates = [p for p in ROOT.rglob('*') if p.is_file() and any(token in p.name.lower() for token in ['manifest', 'summary', 'outputs', 'results'])]
print('candidate_output_file_count=', len(output_candidates))
for p in output_candidates[:80]:
    print(p)


In [ ]:
package_name = 'cab_compact20_3model_provider_outputs.zip'
package_path = ROOT / package_name
source_dirs = [p for p in [ROOT / 'outputs', ROOT / 'results', ROOT / 'reports', ROOT / 'data' / 'results'] if p.exists()]
if not source_dirs:
    print('No output directories found yet; packaging skipped to avoid fabricating artifacts.')
else:
    manifest = {'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'sources': [str(p) for p in source_dirs], 'files': []}
    with zipfile.ZipFile(package_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root in source_dirs:
            for file in root.rglob('*'):
                if file.is_file() and file.stat().st_size < 200 * 1024 * 1024:
                    arc = file.relative_to(ROOT)
                    zf.write(file, arc.as_posix())
                    manifest['files'].append({'path': arc.as_posix(), 'sha256': hashlib.sha256(file.read_bytes()).hexdigest(), 'bytes': file.stat().st_size})
        zf.writestr('manifest.json', json.dumps(manifest, indent=2, sort_keys=True))
    print('package_created=', package_path)


## Import Instructions

After approved execution, run `python3 scripts/check_evidence_safety.py && python3 scripts/check_claim_ledger.py`, then reconcile run index with `python3 scripts/check_run_index.py`.

## Troubleshooting

- Missing Kaggle input path: open the dataset panel and confirm the mounted path, then rerun the input validation cell.
- Wrong working directory: set `ROOT` to the extracted project root before executing commands.
- No GPU detected: confirm accelerator settings, restart session, and rerun the setup cell.
- CUDA out of memory: reduce batch size/shards, run one provider/model at a time, or resume from completed shard manifests.
- Missing package: install only the package named by the project requirements, then rerun import checks.
- Internet disabled: use preuploaded datasets/model caches; do not silently download unavailable models.
- File permission issues: write to `/kaggle/working` or the Colab working directory and package from there.
- Partial outputs: run output validation, preserve completed shards, and rerun only missing shards if the project importer permits it.
- Duplicate chunk imports: compare manifest hashes before importing and refuse mismatched duplicate run tags.
- Corrupted zip: compute sha256, unzip with `testzip`, and rerun packaging from raw outputs if validation fails.
- Session timeout: keep checkpoint/manifest files in the working output directory and resume from the last validated shard.
